## 🐈 모델 구축

머신러닝 모델을 구축하는 과정에서 가장 먼저 결정해야 할 사항 중 하나는 올바른 유형의 모델을 선택하는 것입니다 — 예측(predictive) 모델 또는 생성(generative) 모델. 예측 모델은 입력 데이터를 기반으로 결과를 예측하는 데 초점을 맞추고, 생성 모델은 새로운 샘플을 생성하기 위해 데이터의 기본 분포를 학습하는 것을 목표로 합니다.

우리의 사용 사례는 예측 머신러닝으로 분류됩니다. 예측 모델을 구축하는 방법은 많고 다양합니다. 우리의 사용 사례를 위해 신경망을 선택했습니다. 신경망은 더 나은 일반화 능력을 가지고 있고, 복잡한 패턴을 처리할 수 있으며, 더 표현력이 뛰어나기 때문입니다.

## 🐠 패키지 설치 및 불러오기

마찬가지로 노트북을 개발하면서 패키지들을 설치하고 불러와야 합니다.

이것은 몇 분 정도 걸릴 수 있으며, `pip`에서 오류가 발생하더라도 걱정하지 않아도 됩니다. 어쨌든 모든 것이 잘 실행될 것입니다.

In [ ]:
!pip -q install keras "tensorflow==2.15.1" "tf2onnx" "onnx" "seaborn" "onnxruntime"

In [ ]:
from pathlib import Path
import pickle
import os
import logging, warnings
import random

# 경고 억제하기
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
logging.getLogger('tensorflow').setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
# from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization, Activation, Input, Concatenate
from tensorflow.keras.models import Model
import tf2onnx
import onnx
import tensorflow as tf

from sklearn.metrics import confusion_matrix
import seaborn as sns
from matplotlib import pyplot as plt
import onnxruntime as rt

# 몇 가지 시드값 설정하기
SEED = 42
# np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
# random.seed(SEED)
# os.environ['PYTHONHASHSEED'] = str(SEED)

# 📦 데이터 불러오기

우리는 두 데이터셋을 다시 불러와 이전과 마찬가지로 병합하고, NA 컬럼을 제거한 후 입력 데이터와 출력 데이터를 선택합니다.

입력 데이터(X)는 각 곡의 특성을 포함하는 피처 행렬입니다.

출력 데이터(y)는 모델이 예측하려는 목표 변수입니다. 이 경우 y는 '국가' 컬럼으로, 곡이 인기 있는 국가를 나타냅니다. 모델은 X의 곡 특성들을 기반으로 국가를 예측하는 것을 학습합니다.

In [ ]:
song_properties = pd.read_parquet('https://github.com/rhoai-mlops/jukebox/raw/refs/heads/main/99-data_prep/song_properties.parquet')
song_rankings = pd.read_parquet('https://github.com/rhoai-mlops/jukebox/raw/refs/heads/main/99-data_prep/song_rankings.parquet')
song_properties.keys()

In [ ]:
# 데이터셋에서 결측값(NaN)을 제거합니다
song_rankings = song_rankings.dropna()

모델을 학습하기 전에 데이터를 적절히 준비해야 합니다:  

1. 목표 변수 인코딩하기 (`y`):  우리의 목표 변수(`y`)는 곡이 인기 있는 **국가**입니다. 하지만 머신러닝 모델은 텍스트 레이블이 아닌 숫자를 다룹니다. Label Encoder를 사용하여 국가 이름을 숫자 값으로 변환합니다.  

2. 데이터 분할하기: 데이터셋을 학습, 검증, 테스트 세트로 분할하여 모델이 잘 일반화되도록 합니다.  
   - 학습 세트 (`X_train, y_train`): 모델을 학습하는 데 사용됩니다.  
   - 검증 세트 (`X_val, y_val`): 모델을 미세 조정하고 과적합을 방지하는 데 도움이 됩니다.  
   - 테스트 세트 (`X_test, y_test`): 모델이 본 적 없는 데이터에 대해 얼마나 잘 작동하는지 평가하기 위해 가장 끝에서 사용됩니다.  

3. 피처 스케일링: 입력 피처들(예: `duration_ms`, `danceability`, `loudness`)이 서로 다른 범위를 가지고 있으므로, `MinMaxScaler`를 사용하여 0과 1 사이로 스케일링합니다. 이는 큰 값을 가진 피처가 작은 피처를 지배하는 것을 방지하고 모델이 효율적으로 학습하도록 돕습니다.

In [ ]:
# X는 모델을 학습시킬 입력 피처이고, y는 모델이 예측할 출력 피처입니다.
X = song_rankings.merge(song_properties, on='spotify_id', how='left')
X = X[['is_explicit', 'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']]
y = song_rankings['country']

# 레이블 인코더를 사용하여 국가 코드 대신 숫자를 얻습니다
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_one_hot = tf.keras.utils.to_categorical(y_encoded)

# 데이터를 학습 및 테스트 세트로 분할하여 학습된 모델을 테스트할 데이터를 확보합니다.
X_train, X_test, y_train, y_test = train_test_split(X, y_one_hot, test_size = 0.2, shuffle = False, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train,y_train, test_size = 0.2, stratify = y_train, random_state=SEED)

# 데이터를 스케일링하여 평균을 제거하고 단위 분산을 가지도록 합니다. 데이터는 -1과 1 사이가 되며, 이는 
# 무작위하고 잠재적으로 큰 값보다 모델이 학습하기 훨씬 쉽습니다.
# 스케일러를 학습 데이터에만 적합시키는 것이 중요합니다. 그렇지 않으면 변수의 전 지구적 
# 분포에 대한 정보(테스트 세트의 영향을 받음)가 학습 세트로 유출됩니다.
scaler = MinMaxScaler()
scaled_x_train = pd.DataFrame(scaler.fit_transform(X_train), index=X_train.index, columns=X_train.columns)
scaled_x_val = pd.DataFrame(scaler.transform(X_val), index=X_val.index, columns=X_val.columns)
scaled_x_test = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns).astype(np.float32)

# 🛟 모델 저장을 위한 준비

신경망을 구축하고 모델을 학습하기 전에, 결과 아티팩트를 저장할 환경을 준비해봅시다. 

모델 아티팩트를 S3 버킷의 models/model-name/version/ 폴더에 저장해야 버전 관리를 할 수 있습니다.

In [ ]:
# 신경망을 구축하고 모델을 학습하기 전에 모델 아티팩트를 저장할 로컬 디렉토리를 만듭니다
Path("models/jukebox/1/artifacts").mkdir(parents=True, exist_ok=True)

with open("models/jukebox/1/artifacts/scaler.pkl", "wb") as handle:
    pickle.dump(scaler, handle)

with open("models/jukebox/1/artifacts/label_encoder.pkl", "wb") as handle:
    pickle.dump(label_encoder, handle)

with open("models/jukebox/1/artifacts/y_test.pkl", "wb") as handle:
    pickle.dump(y_test, handle)

X_train.to_parquet("models/jukebox/1/artifacts/X_train.parquet")
X_test.to_parquet("models/jukebox/1/artifacts/X_test.parquet")

# 🚀 모델 구축

아래의 코드는 마치 스마트 보조자, 즉 모델을 만드는 것과 같습니다. 이 모델은 곡의 특성(특징)을 기반으로 어떤 국가가 그 곡을 좋아할지 추측하는 것을 학습합니다. 이러한 특성들을 처리하기 위해 우리의 모델은 여러 층의 "뉴런"을 통과시킵니다. 이것들은 우리 뇌에서처럼 작동하며, 층과 뉴런이 많을수록 학습 능력이 더 커집니다.

마지막에 우리의 모델은 학습한 내용을 사용하여 곡을 가장 즐길 만한 국가들을 예측합니다. 

마지막으로 우리의 모델이 이러한 추측을 얼마나 잘 하고 있는지 확인합니다!

In [ ]:
# 각 개별 입력에 대해 밀집 층을 갖습니까?
inputs = [Input(shape=(1,), name=name) for name in X.columns]
concatenated_inputs = Concatenate(name="input")(inputs)
x = Dense(32, activation='relu', name="dense_0")(concatenated_inputs)
x = Dense(64, name="dense_1")(x)
x = Activation('relu')(x)
x = Dense(128, name="dense_2")(x)
x = Activation('relu')(x)
x = Dense(256, name="dense_3")(x)
x = Activation('relu')(x)
output = Dense(y_one_hot.shape[1], activation='softmax', name="dense_4")(x)
model = Model(inputs=inputs, outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy', 'Precision', 'Recall'])

# 🪿 모델 요약

이제 `model.summary()`를 실행해봅시다. 이 함수는 당신의 음악 추천 보조자의 청사진을 출력합니다. 

실행하면 긴 테이블이 표시되어 곡 피처가 국가 예측으로 변환되기 전에 통과하는 모든 다양한 처리 단계를 보여줍니다(출력 길이에 두려워하지 마세요!).

표는 당신의 음악 뇌 🧠의 각 층을 보여줍니다. 곡의 특성을 받는 방법부터 시작하여, 다양한 "생각 층"(그 `Dense`와 `Activation` 부분들)을 통해 처리하는 방법, 그리고 마지막으로 국가 예측을 생성하는 방법을 보여줍니다.

각 층에 대해 그 이름, 정보 흐름의 형태(예: 몇 개의 숫자가 처리되는지), 그리고 예측을 더 잘하기 위해 조정할 수 있는 "학습 노브"(`parameters`)의 개수가 표시됩니다 🤓

맨 아래에는 이러한 학습 노브의 총 개수가 표시되며, 이는 당신의 추천 시스템이 얼마나 복잡한지를 나타냅니다. 더 많은 수는 당신의 시스템이 다양한 국가에서의 음악 선호도의 더 복잡한 패턴을 학습할 수 있음을 의미합니다. (이것이 LLM 대화에서도 매개변수 개수에 대해 자주 언급되는 이유입니다)

이 요약은 뒤에 숨겨진 모든 수학을 이해할 필요 없이 음악 추천 시스템의 복잡성을 이해하는 데 도움이 될 수 있습니다.

In [ ]:
model.summary()

# 🏃 모델 학습

이제 우리의 스마트 보조자를 학습시켜 곡의 특성을 기반으로 어떤 국가가 그 곡을 좋아할지 예측하도록 합니다. 2번의 에포크(epoch) 동안 학습 데이터에서 학습하도록 설정합니다. 에포크는 전체 데이터셋을 본 횟수를 의미합니다. 각 라운드에서 곡의 특성(scaled_x_train)과 국가 레이블(y_train)을 봅니다. 또한 각 에포크 후 검증 데이터셋(X_val과 y_val)에서 예측합니다. 이는 아직 학습하지 않은 데이터에서 얼마나 잘 작동하는지를 보기 위함입니다.

위의 셀에서 데이터를 3개로 분할한 이유가 바로 이것입니다 :)

학습이 완료되면 모델이 예측을 할 준비가 되었음을 알려주는 메시지를 출력합니다!

In [ ]:
feature_names = list(X_train.columns)

train_features = [scaled_x_train[[name]].to_numpy() for name in feature_names]
val_features = [scaled_x_val[[name]].to_numpy() for name in feature_names]

train_feature_dataset = tf.data.Dataset.zip(tuple(
    tf.data.Dataset.from_tensor_slices(f) for f in train_features
))
val_feature_dataset = tf.data.Dataset.zip(tuple(
    tf.data.Dataset.from_tensor_slices(f) for f in val_features
))

train_dataset = tf.data.Dataset.zip((train_feature_dataset, tf.data.Dataset.from_tensor_slices(y_train)))
val_dataset = tf.data.Dataset.zip((val_feature_dataset, tf.data.Dataset.from_tensor_slices(y_val)))

train_dataset = train_dataset.shuffle(buffer_size=len(y_train), seed=42, reshuffle_each_iteration=False)
train_dataset = train_dataset.batch(32)
val_dataset = val_dataset.batch(32)

주의: 실행하고 싶은 에포크의 수를 변경할 수 있습니다. 수를 늘리면 각 실행을 위해 더 많은 메모리가 필요하지만, 에포크의 수를 계속 늘리면 모델이 더 나빠질 수 있습니다. 에포크 수를 늘린 후 커널을 다시 시작해야 합니다.

In [ ]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset, 
    epochs=2, 
    verbose=True
)

# 🫡 모델 저장

여기서 우리는 학습된 곡 예측 모델을 ONNX라는 인기 있는 형식으로 변환합니다.  
또한 나중에 사용할 수 있도록 원본 Keras 모델을 저장하여 더 쉽게 검사할 수 있습니다.

In [ ]:
input_signature = [tf.TensorSpec(i.shape, i.dtype, i.name) for i in model.inputs]
model.output_names = ['output']
onnx_model_proto, _ = tf2onnx.convert.from_keras(model, input_signature)
onnx.save(onnx_model_proto, "models/jukebox/1/model.onnx")

model.save('models/jukebox/1/model.keras')

### 퀴즈 시간 🤓

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('../.dontlookhere/'))
try: from quiz2 import *
except: pass

In [ ]:
try: quiz_model() 
except: pass

In [ ]:
try: quiz_nn()
except: pass

# 🔥 테스트를 위한 모델 불러오기

여기서 우리는 어떤 국가가 곡을 좋아할지 예측하기 위해 모델을 불러옵니다. 모델에 세션을 열고 테스트 데이터(X_test)를 입력합니다. 모델은 예측을 출력하고 국가들을 식별합니다.

정확도는 예측된 국가를 실제 국가와 비교하여 계산됩니다. 우리는 데이터에 실제 답변을 가지고 있으므로 정확도 메트릭을 얻을 수 있습니다.

또한 혼동 행렬을 만들어 예측 결과를 시각화하고 히트맵을 사용하여 모델의 예측이 실제 레이블과 얼마나 일치하는지를 보여줍니다. 우리는 예측된 국가가 실제 국가와 가능한 한 자주 같기를 원하며, 이는 대각선에 어두운 정사각형이 있는 것으로 시각화됩니다. 예를 들어 국가 0이 종종 국가 0으로 예측됩니다(다른 국가로 예측되지 않음).

즉, 대각선이 어두울수록 좋은 예측에 가까워집니다.

In [ ]:
test_inputs = {name: scaled_x_test[[name]].to_numpy() for name in X_test.columns}

In [ ]:
sess = rt.InferenceSession("models/jukebox/1/model.onnx", providers=rt.get_available_providers())
output_name = sess.get_outputs()[0].name
y_pred_temp = sess.run([output_name], test_inputs)
y_pred_temp = y_pred_temp[0]
y_pred_argmax = np.argmax(y_pred_temp, axis=1)

In [ ]:
y_test_argmax = np.argmax(y_test, axis=1)

In [ ]:
accuracy = np.sum(y_pred_argmax == y_test_argmax) / len(y_pred_argmax)
print("정확도: " + str(accuracy))

c_matrix = confusion_matrix(y_test_argmax,y_pred_argmax)
ax = sns.heatmap(c_matrix, cmap='Blues')
ax.set_xlabel("Prediction")
ax.set_ylabel("Actual")
ax.set_title('Confusion Matrix')
plt.show()

그리고 이제 우리는 모델을 S3 버킷에 저장해야 이 노트북 외부에서 사용할 수 있습니다. [2-save_model.ipynb](2-save_model.ipynb)을 열어주세요 :)